In [1]:
# Cell 1 docstring and import statements
"""
This notebook simulates wind farm operation under a selected control strategy,
compares turbine-level metrics to a baseline budget, and generates summary
metrics for evaluation and comparison.

Supported Modes:
- Evaluation: "timeseries" or "distribution"
- Control strategy: "fixed", "lookup", "timeseries", "optimized" (future)
- Rule stacking: apply rule-based conditions (e.g. bats, revenue threshold, curtailment) on top of WF flow control

Inputs:
- Wind data: time series or joint distribution (with wsp, TI, wdir)
- Control definitions: yaw and power setpoints from any source
- Rules: optional shutdown or control override conditions
- Surrogate model: computes loads and power per turbine
- Price input: DA spot or fixed PPA, aligned with wind input mode
- Baseline reference: JSON with precomputed budget metrics

Outputs:
- turbine_metrics.parquet: per-turbine aggregated output
- compare_to_baseline.json: normalized usage vs baseline
- farm_metrics.json: farm-wide KPIs
- settings.json: full configuration metadata including control stack

Control architecture:
- All control strategies and rules are defined as modular functions
- Controllers can be layered and gated by rules
- Rules can suppress WF control or trigger shutdown based on conditions
- Entire control stack is dynamically composed and logged

Intended Usage:
- Evaluate alternative control strategies in simulation
- Analyze fatigue, power, revenue, and actuator usage
- Compare controlled operation against baseline budgets
- Extend with custom metrics and rules

"""


import os
import numpy as np
import pandas as pd
import json
from pathlib import Path

from input_handling import (
     load_wind_timeseries, load_joint_distribution,
    calculate_bin_edges, validate_lookup_turbine_ids,load_surrogate_models,load_baseline_inputs,load_aux_inputs
)
from simulation_engine import simulate_block, simulate_distribution
from control_interface import constant_control, lookup_controller, rule_controller, combine_controllers, shutdown_rule, wffc_activation_rule,override_controller
# from postprocessing import compare_to_baseline, generate_farm_metrics
from postprocessing import aggregate_farm_budget, save_results
from farm_setup import initialize_HKN_pywake_farm




ModuleNotFoundError: No module named 'input_handling'

In [2]:
pwd

'C:\\'

In [ ]:
# Cell 2: Configuration and Input Setup

# Can be: "timeseries" or "distribution"
eval_mode = "distribution" 

# Input files
wind_price_TS_file = "data/timeseries/HKNB_timeseries_full_filled_small_gaps_only_TI_boost.csv"
# distribution_file = "distributions/joint_3D_distribution_price.csv" 
distribution_file = "distributions/HKNB_Weib_WSWD_TI_dist_3deg_clip_TIboost.csv" 

aux_files= [
    # "data/aux_data/aux_bats_dynamic.csv",
            # "data/aux_data/aux_misc.csv",
            ]  # List of auxiliary input files

# baseline_dir = "baselines/TS_FA_HKN_v3_TIboost"
baseline_dir = "baselines/Dist_HKN_WSWD_TI_dist_3deg_TIboost"
 # If True, load baseline inputs from the file to feed to the controllers. Only relevant for the controller input in timeseries mode
use_baseline_inputs = False  

# Input resolution: "10min" or "1h" releevant for timeseries only
input_resolution = "10min" # '10min' or  '1h'
# Basis lifetime for damage metrics
lifetime_years = 25  # Needed for distribution modes
layout_file = 'layout/HKN_codesign_8649_layout.csv'
# layout_file = 'layout/HKN_dummy_3.csv'
# Initialize farm from a function (pywake or floris)
farm = initialize_HKN_pywake_farm(layout_file) # load pywake or flroris farm setup
lookup_table_cntrl_file =r'data/controller_inputs/HKN_codesign_8649_yaw_lookup.csv' # CSV file with lookup table control inputs only relevant when lookup is active
# lookup_table_cntrl_file =r'data/controller_inputs/dummy_lookup_controller_V2.csv' 
# farm = initialize_single_turbine_farm_IEA22()

# Output
case_name = "HKN_codesign_8649_WSWD_TI_dist_3deg_TIboost"  # Name of the case for output files
output_path = Path(f"results/{case_name}")
output_path.mkdir(parents=True, exist_ok=True)

#Flags to save time series or per bin outputs
return_turbine_timeseries=True # If True, save turbine-level time series in the output parquet file
return_turbine_timeseries_cum=True # If True, save cumulative turbine-level time series in the output parquet file
return_farm_timeseries=True # If True, save farm-level time series in the output parquet file
return_bin_outputs=True  # If True, save bin outputs (e.g., actuator counts) in the output parquet file


# Price input (optional)
use_prices = False # If false, orice not used and the next 2 parameters are ignored
price_type = "DA_spot"  # "DA_spot" or "fixed_PPA"
fixed_price_value = 100.0  # EUR/MWh, used if price_type == "fixed_PPA"

# Wind speed operational parameters
wsp_cut_in = 3
wsp_cut_off = 25


# Surrogate options
model_path_base="models/RA_IEA22_DTU" # Path to surrgoate keras models
scaler_path_base="models/RA_IEA22_DTU" # Path to surrogate scalers
use_sector_average = False  # If True, use sector average based surrogate
supports_operating_modes = False # If True, the surrogate supports operating modes besides normal operation (idling, startup, shutdown)

channels = [
    "RA_ADC", "RA_BRM","RA_ebrm", "RA_fbrm", "RA_blade_torsion", "RA_tbfa",
    "RA_tbss", "RA_TBM", "RA_TB_torsion","RA_ttfa", "RA_TTM","RA_gsb_l10","RA_rsb_l10","RA_shaft_mx_mb_fixed","RA_shaft_mz_mb_fixed"
]
channel_processing_specs = {
    # DEL-based damage metrics (DEL → D)
    "RA_tbfa": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_tbss": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_TBM": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_TB_torsion": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_TTM": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_ttfa": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_BRM": {"type": "damage", "wohler": 10, "nref": 600},
    "RA_ebrm": {"type": "damage", "wohler": 10, "nref": 600},
    "RA_fbrm": {"type": "damage", "wohler": 10, "nref": 600},
    "RA_blade_torsion": {"type": "damage", "wohler": 10, "nref": 600},  
    "RA_shaft_mx_mb_fixed": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_shaft_mz_mb_fixed": {"type": "damage", "wohler": 4, "nref": 600},
    # Harmonic mean channels (bearing life rates)
    "RA_gsb_l10": {"type": "harmonic"},
    "RA_rsb_l10": {"type": "harmonic"},
    # Simple accumulators (e.g. actuators, custom metrics)
    "RA_ADC": {"type": "sum"},  # actuator counts (e.g., pitch)
    
    # Handled internally — not in surrogates
    # "YawTravel": {"type": "sum"},
    # "Energy": {"type": "sum"},
    # "Revenue": {"type": "sum"},  # only if prices are used
}


In [ ]:
# Cell 3: Controller strategies and Rules definition

# Controllers setup
control_mode = "lookup"     # "fixed", "lookup", "timeseries"  base controller WFFC
use_rule_based = False  # False to skip rules
use_override = False  # False to skip overrides

# FIXED
fixed_control = {    # only used with fixed control logic
    "yaw": np.full(34, 0),
    "power":np.full(34, 100),
    "shutdown": np.full(34, False),
}

#LOOKUP
# lookup_table_cntrl_file =r'data/controller_inputs/dummy_lookup_controller.csv'
cntrl_lookup_df = pd.read_csv(lookup_table_cntrl_file) # only used with lookup control logic


# RULE BASED 
rule_flags = {
    # "shutdown_rule_1": {
    #     "groups": [
    #         {
    #             "conditions": [
    #                 {"var": "wsp", "op": "<", "value": 4},
    #                 {"var": "TI", "op": ">", "value": 0.15}
    #             ],
    #             "logic": "AND"
    #         },
    #         {
    #             "conditions": [
    #                 {"var": "wsp", "op": ">", "value": 18},
    #                 {"var": "price", "op": "<", "value": 60}
    #             ],
    #             "logic": "AND"
    #         }
    #     ],
    #     "groups_logic": "OR"
    # },

    # "shutdown_rule_wspOnly": {
    #     "groups": [
    #         {"conditions": [{"var":"wsp","op":"<","value":6.0}], "logic":"AND"},          
    #     ],
    #     "groups_logic": "OR"
    # },

    # "shutdown_rule_wspTIcond": {
    #     "groups": [
    #         {"conditions":[{"var":"wsp","op":"<","value":6.0},{"var":"TI","op":">","value":0.1}], "logic":"AND"},
    #         {"conditions":[{"var":"wsp","op":">","value":18.0},{"var":"TI","op":">","value":0.05}], "logic":"AND"},
    #     ],
    #     "groups_logic": "OR",
    # },
    
    # "shutdown_rule_REVonly": {
    #     "groups": [
    #         {"conditions":[{"var":"baseline.revenue","op":"<","value": 1000.0}], "logic":"AND"},
    #     ],
    #     "groups_logic": "OR",
    # }

    #    "shutdown_rule_EnergyOnly": {
    #     "groups": [
    #         {"conditions":[{"var":"baseline.energy","op":"<","value": 15.0}], "logic":"AND"},
    #     ],
    #     "groups_logic": "OR",
    # }

    "wf_activation_rule_wspTIonly": {
        "groups": [
            {"conditions":[{"var":"wsp","op":">","value":7.5},{"var":"TI","op":"<","value":0.2}], "logic":"AND"},
        ],
        "groups_logic": "OR",
    },
    
    #  "wf_activation_rule_REV_energy": {
    #     "groups": [
    #         {"conditions":[
    #             {"var":"baseline.revenue","op":"<","value":1200.0},
    #             {"var":"baseline.energy","op":"<","value": 12.0}
    #         ], "logic":"OR"},
    #     ],
    #     "groups_logic": "OR",
    # },


     "shutdown_rule_Combined": {
        "groups": [
            {"conditions":[
                # {"var":"baseline.energy","op":"<","value":45.5},
                {"var":"baseline.revenue","op":"<","value":1400.0}
            ], "logic":"AND"},
            {"conditions":[
                {"var":"wsp","op":">=","value":18.0}
            ], "logic":"AND"}
        ],
        "groups_logic":"OR",
    }


    # "shutdown_rule_REV": {
    #     "groups":[{"conditions":[{"var":"baseline.revenue","op":"<","value":1400.0}], "logic":"AND"}],
    #     "groups_logic":"OR",
    # },

    # "wf_activation_rule_1": {
    #     "groups": [
    #         {
    #             "conditions": [
    #                 {"var": "TI", "op": ">", "value": 0.2}
    #             ],
    #             "logic": "AND"
    #         }
    #     ],
    #     "groups_logic": "OR"
    # }
}

# Combine controller and define the hierarchy of the control stack
rule_library = {
    "shutdown_rule_Combined": shutdown_rule,
    "wf_activation_rule_wspTIonly": wffc_activation_rule
}


# Override controlers, final layer
override_flags = {
    # "bat_dynamic_mock": {"activity_keys": ("bat_act1", "bat_act2")},
    "yaw_band_override": {"wsp_min": 8.0, "wsp_max": 9.0, "yaw_angle": 29.9, "target_turbines": [1,2,3]},
    # add more override instances here as needed
}


In [ ]:
# Cell 4: Load Inputs and Prepare Simulation

# Load surrogate models and metadata
surrogate,surrogate_metadata = load_surrogate_models(
    model_path_base=model_path_base,
    scaler_path_base=scaler_path_base,
    channels=channels,
    use_sector_average=use_sector_average,
    supports_operating_modes=supports_operating_modes
)
# Initialize farm and control functions
layout = farm["layout_df"]

if eval_mode == "timeseries":
    wind_data = load_wind_timeseries(wind_price_TS_file, input_resolution,require_price=use_prices)
    # Master timeline for alignment
    expected_timestamps = pd.DatetimeIndex(pd.to_datetime(wind_data["timestamp"]))
    if use_baseline_inputs:
        baseline_df = load_baseline_inputs(baseline_dir, expected_timestamps)
    else:
        baseline_df = None

    if  aux_files is not None and len(aux_files) > 0:
        aux_df = load_aux_inputs(expected_timestamps, aux_files)
    else:
        aux_df = None

    if use_prices and price_type == "DA_spot":
        # Price comes from the same wind_data now
        if "price" not in wind_data.columns:
            raise ValueError("Expected price column not found in wind timeseries.")
        price_series = wind_data["price"].values
    elif use_prices and price_type == "fixed_PPA":
        price_series = np.full(len(wind_data), fixed_price_value)
    else:
        price_series = None

elif eval_mode == "distribution":
    distribution = load_joint_distribution(distribution_file)
    if use_prices:
        if "price" not in distribution.columns:
            raise ValueError("Distribution is missing price information.")
        price_per_bin = distribution["price"].values if price_type == "DA_spot" else np.full(len(distribution), fixed_price_value)
    else:
        price_per_bin = None

else:
    raise ValueError("Invalid eval_mode")

# Check that the lookup table has values for all turbines in the layout
if control_mode == "lookup":
    validate_lookup_turbine_ids(cntrl_lookup_df, layout)


In [ ]:
# Cell 5: build control stack

# Create base control (e.g., lookup, fixed)
if control_mode == "fixed":
    base_control = constant_control(
        yaw=fixed_control["yaw"],
        power=fixed_control["power"],
        shutdown=fixed_control["shutdown"]
    )
elif control_mode == "lookup":
    # load lookup table and bin definitions
    
    bin_specs = calculate_bin_edges(cntrl_lookup_df)
    
    base_control = lookup_controller(cntrl_lookup_df, bin_specs,lookup_table_cntrl_file)
else:
    base_control = constant_control()  # fallback for dummy

# Add rules
if use_rule_based:
    rule_wrappers = rule_controller(rule_flags, rule_library)
else:
    rule_wrappers = []
    
# Add override controllers
if use_override:    
    override_wrappers = override_controller(override_flags)
else:
    override_wrappers = []

control_fn = combine_controllers(base_control, rule_wrappers,override_wrappers)


In [ ]:
#Cell 6: Main Simulation 
if eval_mode == "timeseries":
    result_df,turb_ts_inst,turb_ts_cum,farm_ts = simulate_block(
            wind_data=wind_data,
            farm=farm,
            surrogate=surrogate,
            control_fn=control_fn,
            resolution=input_resolution,
            use_sector_average=use_sector_average,  
            supports_operating_modes=supports_operating_modes,
            channel_processing_specs=channel_processing_specs,
            wsp_min=wsp_cut_in,
            wsp_max=wsp_cut_off,
            price_series=price_series,
            baseline_df=baseline_df,       
            aux_df=aux_df, 
            return_turbine_timeseries=return_turbine_timeseries,
            return_turbine_timeseries_cum= return_turbine_timeseries_cum,
            return_farm_timeseries= return_farm_timeseries,
        )
    bin_values = None
elif eval_mode == "distribution":
    # distribution = load_joint_distribution(distribution_file, require_price=use_prices)
    
    if use_prices:
        if "price" not in distribution.columns:
            raise ValueError("Distribution must include 'price' column if prices are used.")
        price_type = price_type  # just for clarity
    else:
        price_type = None  # Ensures simulate_distribution knows prices are off


    result_df, bin_df = simulate_distribution(distribution,
                          farm,
                          surrogate,
                          control_fn,
                          channel_processing_specs,
                          lifetime_years = lifetime_years,
                          wsp_min= wsp_cut_in,
                          wsp_max= wsp_cut_off,
                          supports_operating_modes = supports_operating_modes,
                          use_sector_average = use_sector_average,
                          use_prices=use_prices,
                          price_type=price_type,
                          fixed_price_value=fixed_price_value,
                          return_bin_outputs=return_bin_outputs)
    turb_ts_inst = None
    turb_ts_cum = None
    farm_ts = None
    bin_values = bin_df
    



In [ ]:
# Cell 7: Post-Processing and Comparison
# with open(baseline_dir) as f:
#     baseline_budget = json.load(f)

# usage_metrics = compare_to_baseline(turbine_metrics, baseline_budget)
# farm_metrics = generate_farm_metrics(turbine_metrics)

# Global max per channel and total 
# Include all load and actuator(global max among turbines), power and revenue (sum of all turbines) metrics
fatigue_budget = aggregate_farm_budget(result_df)

control_description = control_fn.describe() if hasattr(control_fn, 'describe') else {"type": control_mode}

settings = {
    "case_name": case_name,
    "baseline_dir": baseline_dir,
    "eval_mode": eval_mode,
    "layout_file": layout_file,
    "input_resolution": input_resolution if 'timeseries' in eval_mode else None,
    "source": wind_price_TS_file if 'timeseries' in eval_mode else distribution_file,
    "aux_files": aux_files if (eval_mode == "timeseries" and aux_files) else [],
    "baseline_loaded": bool(use_baseline_inputs),
    "baseline_dir": baseline_dir if use_baseline_inputs else None,
    "rule_flags": rule_flags if use_rule_based else {},
    "override_flags": override_flags if use_override else {},
    "num_turbines": len(layout) ,
    "control_mode": control_mode,
    "control_stack": control_description,
    "use_rule_based": use_rule_based,
    "use_override": use_override,
    "price_type": price_type if use_prices else None,
    "price_source": (
        distribution_file if (eval_mode == "distribution" and price_type == "DA_spot")
        else wind_price_TS_file if (eval_mode == "timeseries" and price_type == "DA_spot")
        else "fixed_PPA" if (price_type == "fixed_PPA")
        else None
    ),
    "fixed_price": fixed_price_value if (use_prices and price_type == "fixed_PPA") else None,
    "use_prices": use_prices,
    "wsp_cut_in": wsp_cut_in,
    "wsp_cut_out": wsp_cut_off,
    "flow_type": farm.get("flow_type"),
    "farm_setup": farm.get("setup_summary"),
    "rotor_model": farm.get("rotor_model"),
    "num_turbines": int(farm["layout_df"].shape[0]),
    "surrogate": surrogate_metadata
}

save_results(
    output_path=output_path,
    turbine_metrics=result_df,
    fatigue_budget=fatigue_budget,
    settings=settings,
    farm_metrics=None,
    comparison_to_baseline=None,
    turb_ts_inst =turb_ts_inst,
    turb_ts_cum =turb_ts_cum,
    farm_ts=farm_ts,
    bin_values=bin_values
   )


In [ ]:
# inflow_df.to_parquet("inflow_df_control.parquet", index=False)
print(inflow_df.head())

In [ ]:
# ==== Deep inspect of surrogate BoxDomain objects (standalone) ====
import numpy as np

def _decode_bytes_list(x):
    return [i.decode() if isinstance(i, (bytes, bytearray)) else str(i) for i in x]

def _print_attr(name, obj):
    try:
        arr = np.asarray(obj, dtype=float)
        print(f"    {name}: array shape={arr.shape}, "
              f"min={np.nanmin(arr):.6g}, max={np.nanmax(arr):.6g}")
        print(f"      first row preview: {arr[:1]}")
    except Exception:
        print(f"    {name}: {type(obj)} -> {repr(obj)}")

def _try_extract_bounds(domain):
    """
    Return (lower, upper) if we can discover them, else (None, None).
    Tries a bunch of common attribute/method names seen in 'BoxDomain'–like classes.
    """
    # 1) common attribute pairs
    for lo_name, hi_name in [
        ("lower", "upper"),
        ("lb", "ub"),
        ("low", "high"),
        ("mins", "maxs"),
        ("min", "max"),
    ]:
        lo = getattr(domain, lo_name, None)
        hi = getattr(domain, hi_name, None)
        if lo is not None and hi is not None:
            lo = np.asarray(lo, dtype=float).ravel()
            hi = np.asarray(hi, dtype=float).ravel()
            if lo.shape == hi.shape:
                return lo, hi

    # 2) single 'bounds' attribute (could be (2,n) or (n,2))
    b = getattr(domain, "bounds", None)
    if b is not None:
        b = np.asarray(b, dtype=float)
        if b.ndim == 2:
            if b.shape[0] == 2:
                return b[0], b[1]
            if b.shape[1] == 2:
                return b[:, 0], b[:, 1]

    # 3) dict-like export
    for meth in ("to_dict", "as_dict"):
        f = getattr(domain, meth, None)
        if callable(f):
            try:
                d = f()
                # try the same key pairs in the dict
                for lo_name, hi_name in [
                    ("lower","upper"), ("lb","ub"), ("low","high"),
                    ("mins","maxs"), ("min","max"), ("bounds","bounds")
                ]:
                    if lo_name in d and hi_name in d:
                        lo = np.asarray(d[lo_name], dtype=float).ravel()
                        hi = np.asarray(d[hi_name], dtype=float).ravel()
                        if lo.shape == hi.shape:
                            return lo, hi
            except Exception:
                pass

    # 4) array-like export
    for meth in ("to_array", "to_numpy", "as_array", "numpy"):
        f = getattr(domain, meth, None)
        if callable(f):
            try:
                arr = np.asarray(f(), dtype=float)
                if arr.ndim == 2:
                    if arr.shape[0] == 2:
                        return arr[0], arr[1]
                    if arr.shape[1] == 2:
                        return arr[:, 0], arr[:, 1]
            except Exception:
                pass

    # 5) get_bounds method
    f = getattr(domain, "get_bounds", None)
    if callable(f):
        try:
            gb = f()
            gb = np.asarray(gb, dtype=float)
            if gb.ndim == 2:
                if gb.shape[0] == 2:
                    return gb[0], gb[1]
                if gb.shape[1] == 2:
                    return gb[:, 0], gb[:, 1]
        except Exception:
            pass

    return None, None

print("\n=== Surrogate BoxDomain introspection ===")
for mname, model in surrogate.items():
    print(f"\n--- {mname} ---")
    print(f"model type: {type(model)}")

    in_names = getattr(model, "input_names", [])
    in_names = _decode_bytes_list(in_names) if len(in_names) else []
    print("input_names:", in_names)

    dom = getattr(model, "domain", None)
    print("domain type:", type(dom))

    if dom is None:
        print("  No domain present.")
        continue

    # Dump available attributes (short list)
    dom_attrs = [a for a in dir(dom) if not a.startswith("_")]
    print("domain attributes:", dom_attrs[:40], ("..." if len(dom_attrs) > 40 else ""))

    # Try common bound extractors
    lo, hi = _try_extract_bounds(dom)
    if lo is not None and hi is not None:
        print("Resolved bounds (per input):")
        for i, nm in enumerate(in_names[:len(lo)]):
            print(f"  {nm:>14s}: {lo[i]: .6g}  →  {hi[i]: .6g}")
    else:
        print("Could not auto-resolve numeric bounds. Raw attribute previews (best-effort):")
        # Print a few likely fields if they exist
        for cand in ("lower","upper","lb","ub","low","high","mins","maxs","min","max","bounds"):
            if hasattr(dom, cand):
                _print_attr(cand, getattr(dom, cand))

        # Try dict/array exports just to show what's there
        for meth in ("to_dict", "as_dict", "to_array", "to_numpy", "as_array", "numpy", "get_bounds"):
            f = getattr(dom, meth, None)
            if callable(f):
                print(f"  method {meth}():")
                try:
                    out = f()
                    _print_attr(f"{meth}()", out)
                except Exception as e:
                    print(f"    call failed: {e}")
